In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from aqs import cleaning
from aqs.config import CITIES, DATE_FROM, DATE_TO, INTERIM, PROCESSED

In [2]:
cleaning.main()

bangkok: 150 PM2.5 weeks from 64 of 182 sensors; 20 search series
krakow: 134 PM2.5 weeks from 2 of 5 sensors; 19 search series
melbourne: 96 PM2.5 weeks from 8 of 25 sensors; 12 search series
lahore: 123 PM2.5 weeks from 1 of 2 sensors; 11 search series
saved: 8051 long rows, 503 wide rows


In [3]:
sel = pd.read_csv(INTERIM / "sensor_selection.csv")
sel.groupby(["city", "decision"]).size().unstack(fill_value=0).reindex(list(CITIES))

decision,kept,not a reference-grade monitor,under 50% of study days
city,,,
bangkok,64,28,90
krakow,2,2,1
melbourne,8,4,13
lahore,1,1,0


In [4]:
sel[sel["decision"] == "kept"].groupby("city")["valid_share"].describe().reindex(list(CITIES)).round(2)

,count,mean,std,min,25%,50%,75%,max
city,,,,,,,,
bangkok,64.0,0.72,0.11,0.53,0.64,0.70,0.76,0.93
krakow,2.0,0.77,0.07,0.73,0.75,0.77,0.80,0.82
melbourne,8.0,0.57,0.03,0.50,0.57,0.58,0.59,0.59
lahore,1.0,0.76,NaN,0.76,0.76,0.76,0.76,0.76


In [5]:
aq = pd.read_csv(INTERIM / "aq_weekly_all.csv", parse_dates=["week"])
window = len(pd.date_range(cleaning.week_start(pd.Series([DATE_FROM])).iloc[0], DATE_TO, freq="7D"))
coverage = aq.groupby("city").agg(weeks=("week", "size"), first_week=("week", "min"), last_week=("week", "max"),
                                  mean_days_per_week=("n_days", "mean"), mean_sensors=("n_sensors", "mean"))
coverage.insert(1, "weeks_in_window", window)
coverage.insert(2, "coverage_%", (coverage["weeks"] / window * 100).round(0).astype(int))
coverage.reindex(list(CITIES)).round(1)

,weeks,weeks_in_window,coverage_%,first_week,last_week,mean_days_per_week,mean_sensors
city,,,,,,,
bangkok,150,158,95,2022-01-02,2024-12-22,6.8,49.2
krakow,134,158,85,2022-01-02,2024-12-22,6.6,1.9
melbourne,96,158,61,2022-01-02,2024-12-22,6.7,7.6
lahore,123,158,78,2022-01-02,2024-12-22,6.5,1.0


In [6]:
trends = pd.read_csv(INTERIM / "trends_weekly_all.csv", parse_dates=["week"])
trends.groupby(["city", "role"])["term"].nunique().unstack(fill_value=0).reindex(list(CITIES))

role,awareness,health,protection
city,,,
bangkok,4,9,4
krakow,3,9,4
melbourne,1,5,3
lahore,3,5,3


In [7]:
combined = pd.read_csv(PROCESSED / "all_cities_combined.csv", parse_dates=["week"])
print(combined.shape)
combined.groupby("city").agg(weeks=("week", "size"), first_week=("week", "min"), last_week=("week", "max")).reindex(list(CITIES))

(503, 11)


,weeks,first_week,last_week
city,,,
bangkok,150,2022-01-02,2024-12-22
krakow,134,2022-01-02,2024-12-22
melbourne,96,2022-01-02,2024-12-22
lahore,123,2022-01-02,2024-12-22


In [8]:
combined.head()

,week,pm25_mean,city,air_quality,air_purifier,N95_mask,hospital,breathing_difficulty,cough_medicine,eye_drops,chest_pain
0,2022-01-02,34.785714,bangkok,9,28,0,100,NaN,0,46,79
1,2022-01-09,37.085714,bangkok,15,34,91,91,NaN,0,48,64
2,2022-01-16,32.750000,bangkok,9,20,0,74,NaN,0,34,66
3,2022-01-23,25.516667,bangkok,13,22,0,67,NaN,0,42,59
4,2022-01-30,20.992857,bangkok,0,21,0,62,NaN,0,46,31
